In [37]:
from jax import value_and_grad,jit
import optax

import jax

# https://pypi.org/project/get-mnist/
from mnist import fashion_mnist

import jax.numpy as jnp
import numpy as np

from tqdm import tqdm

In [7]:
x, y, xtest, ytest = fashion_mnist()
y = y.astype(float)
ytest = ytest.astype(float)
x = x.reshape(x.shape[0],-1)
xtest = xtest.reshape(xtest.shape[0],-1)

# normalize
x = x/255.0 - 0.5
xtest = xtest/255.0 - 0.5

x.shape

[MNIST] Caching data at /home/sampad/.local/share/FASHION_MNIST
[MNIST] Found http://fashion-mnist.s3-website.eu-central-1.amazonaws.com/train-images-idx3-ubyte.gz in cache.
[MNIST] Found http://fashion-mnist.s3-website.eu-central-1.amazonaws.com/t10k-images-idx3-ubyte.gz in cache.
[MNIST] Found http://fashion-mnist.s3-website.eu-central-1.amazonaws.com/train-labels-idx1-ubyte.gz in cache.
[MNIST] Found http://fashion-mnist.s3-website.eu-central-1.amazonaws.com/t10k-labels-idx1-ubyte.gz in cache.


(60000, 784)

In [42]:
def gen(params, r):
    for p in params[:-1]:
        r = r @ p 
        r = jax.nn.relu(r)
    r = r @ params[-1]
    return r

def dis(params, x):
    for p in params[:-1]:
        x = x @ p 
        x = jax.nn.relu(x)
    x = x @ params[-1]
    x = jax.nn.sigmoid(x)
    return x

def gan(params, r, x, labels):
    gparams,dparams = params
    
    fake = gen(gparams, r)
    real = x

    data = jnp.vstack([real, fake])


    yhat = dis(dparams, data)

    loss = jnp.mean( yhat - labels)
    return loss


dgan = jit(value_and_grad(gan))

In [59]:
gparams = [np.random.randn(x.shape[1],x.shape[1]) for i in range(5)]
dparams = [np.random.randn(x.shape[1],100),np.random.randn(100,50), np.random.randn(50,1)]
params = [gparams, dparams]

In [66]:
random = np.random.randn(*x.shape)

In [67]:
BSIZE = 1000
EPOCHS = 100

In [68]:
goptimizer = optax.amsgrad(learning_rate=1e-3)
doptimizer = optax.amsgrad(learning_rate=1e-3)

gstate = goptimizer.init(params[0])
dstate = doptimizer.init(params[1])

In [69]:
pbar  = tqdm(range(EPOCHS), colour="blue", ncols=100)

disloss, genloss = 0,0
for _ in pbar:
    bids = range(0,x.shape[0],BSIZE)

    for bid in bids:
        xbatch = x[bid:bid+BSIZE]
        rbatch = random[bid:bid+BSIZE]

        #####################
        traingen = False
        if traingen == False:
            real_labels = np.ones(xbatch.shape[0])
            fake_labels = np.zeros( rbatch.shape[0])
            
        elif traingen == False:
            real_labels = np.zeros( xbatch.shape[0])
            fake_labels = np.ones( rbatch.shape[1])
        
        labels = np.hstack([real_labels, fake_labels])
        #####################

        # train discriminator
        disloss, gradient = dgan(params, rbatch, xbatch, labels)
        dis_gradient = gradient[1]
        dupdates, dstate = doptimizer.update(dis_gradient, dstate, params[1])
        optax.apply_updates(params[1], dupdates)

        pbar.set_postfix_str(f"genloss: {genloss*100:.2f}% \t disloss: {disloss*100:.2f}%")
        
    
    for bid in bids:
        xbatch = x[bid:bid+BSIZE]
        rbatch = random[bid:bid+BSIZE]

        #####################
        traingen = True
        if traingen == False:
            real_labels = np.ones(xbatch.shape[0])
            fake_labels = np.zeros( rbatch.shape[0])
            
        elif traingen == True:
            real_labels = np.zeros( xbatch.shape[0])
            fake_labels = np.ones( rbatch.shape[1])
        
        labels = np.hstack([real_labels, fake_labels])
        #####################
        
        # train generator
        genloss, gradient = dgan(params, rbatch, xbatch, labels)
        gen_gradient = gradient[0]
        gupdates, gstate = goptimizer.update(gen_gradient, gstate, params[0])
        optax.apply_updates(params[0], gupdates)
        
        pbar.set_postfix_str(f"genloss: {genloss*100:.2f}% \t disloss: {disloss*100:.2f}%")

        
    

 10%|██▊                         | 10/100 [02:32<22:52, 15.25s/it, genloss: 11.06% 	 disloss: 4.72%]


KeyboardInterrupt: 